# End-to-End ML Pipeline for Customer Churn Prediction

## Overview
This project focuses on building a complete machine learning pipeline for predicting customer churn using the Telco Customer Churn dataset. The objective is to develop a reusable and production-ready workflow that automates data preprocessing, model training, evaluation, and model export.

In this notebook, different preprocessing techniques such as missing value handling, feature scaling, and categorical encoding are implemented using Scikit-learn’s Pipeline and ColumnTransformer APIs. Two machine learning models — Logistic Regression and Random Forest — are trained and evaluated to compare their performance.

Additionally, GridSearchCV is used for hyperparameter tuning to improve model accuracy and identify the best-performing configuration. The final optimized pipeline is exported using Joblib for future deployment and reuse.

This project demonstrates essential machine learning engineering practices including modular pipeline construction, automated preprocessing, model optimization, evaluation using classification metrics, and production-ready model saving techniques.

---

## Objectives
- Build a complete machine learning pipeline using Scikit-learn
- Perform automated preprocessing for numerical and categorical data
- Train and evaluate multiple machine learning models
- Apply hyperparameter tuning using GridSearchCV
- Export the trained pipeline using Joblib
- Develop a reusable and production-ready ML workflow

---

## Technologies Used
- Python
- Pandas
- NumPy
- Scikit-learn
- Joblib
- Google Colab

---

## Dataset
Dataset Used: Telco Customer Churn Dataset

The dataset contains customer demographic information, account details, subscribed services, and churn labels used to predict whether a customer is likely to leave the company.

---

## Evaluation Metrics
The following evaluation metrics are used:
- Accuracy Score
- Classification Report
- Precision
- Recall
- F1-Score

---

## Expected Outcome
By the end of this project, a fully trained and optimized customer churn prediction pipeline will be developed and saved for deployment or future inference tasks.

#Import Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report

import joblib

#Load Dataset

In [3]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#Basic Preprocessing

In [5]:
# Remove customerID
df.drop('customerID', axis=1, inplace=True)

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Convert target column
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


#Separate Features and Target

In [6]:
X = df.drop('Churn', axis=1)
y = df['Churn']

#Identify Numerical and Categorical Columns

In [7]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns

categorical_features = X.select_dtypes(include=['object']).columns

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

Categorical Features:
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


#Create Preprocessing Pipeline

In [8]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

#Split Dataset

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#Logistic Regression Pipeline

In [10]:
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('classifier', LogisticRegression(max_iter=1000))])

#Predictions and Evaluation

In [11]:
y_pred = logistic_pipeline.predict(X_test)

print("Accuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score:
0.8211497515968772

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409



#Random Forest Pipeline

In [12]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier())])

#Random Forest Evaluation

In [13]:
rf_pred = rf_pipeline.predict(X_test)

print("Accuracy Score:")
print(accuracy_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Accuracy Score:
0.7906316536550745

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.65      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409



#GridSearchCV

In [14]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'classifier__max_depth': 10, 'classifier__n_estimators': 100}


#Save Model

In [15]:
joblib.dump(grid_search.best_estimator_, 'customer_churn_pipeline.pkl')

print("Pipeline saved successfully!")

Pipeline saved successfully!


#Download Saved Model

In [16]:
from google.colab import files

files.download('customer_churn_pipeline.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusion

In this project, an end-to-end machine learning pipeline was successfully developed for customer churn prediction using the Telco Customer Churn dataset. The workflow included data preprocessing, feature transformation, model training, evaluation, hyperparameter tuning, and model export using Scikit-learn’s Pipeline API.

Both Logistic Regression and Random Forest models were implemented and evaluated using classification metrics such as accuracy, precision, recall, and F1-score. GridSearchCV was applied to optimize model performance and identify the best hyperparameters for the Random Forest classifier.

The project demonstrates how machine learning pipelines can simplify preprocessing and model deployment by combining all stages into a single reusable workflow. Additionally, exporting the trained pipeline using Joblib makes the solution production-ready and easy to integrate into future applications.

Overall, this task provided practical experience in machine learning engineering, automated workflows, model optimization, and production-focused development practices.